# 🎬 SmartCare+ Guardian: Video Action Recognition Fine-Tuning

## MoViNet/SlowFast Fine-Tuning for Fall Detection & Gait Analysis

**Owner:** Madhushani (Guardian Service)

This notebook fine-tunes a pre-trained video action recognition model (MoViNet-A2) for:
- **Fall Detection** (forward, backward, sideways, collapse)
- **Gait Analysis** (normal gait, arthritic gait, abnormal patterns)
- **ADL Recognition** (walking, sitting, standing, lying)
- **TUG Test** (Timed Up and Go assessment)

### Dataset Structure Expected:
```
dataset/
├── ADL/              # Activities of Daily Living
├── Fall/             # Various fall types
├── arthritis_gait/   # Abnormal gait patterns
├── good_gait/        # Normal gait patterns
└── tug/              # Timed Up and Go test videos
```

### Why MoViNet instead of SlowFast?
- MoViNet-A2 is more efficient and has better TensorFlow Hub support
- Achieves similar accuracy with smaller model size
- Better suited for mobile deployment (SmartCare+ app)

## Section 1: Setup & Install Dependencies

Install required libraries and import modules. Enable mixed precision for faster GPU training.

In [ ]:
# Install required libraries (run this cell first)
!pip install -q tensorflow
!pip install -q tensorflow-hub
!pip install -q opencv-python-headless
!pip install -q scikit-learn
!pip install -q matplotlib seaborn
!pip install -q tqdm

print("✅ All packages installed!")

In [ ]:
# Import libraries
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import cv2
import os
import json
import random
from pathlib import Path
from typing import Tuple, List, Dict, Optional
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Enable mixed precision for faster training on GPU
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

# Check GPU availability
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")
print(f"Mixed precision policy: {mixed_precision.global_policy().name}")

# Set random seeds for reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

## Section 2: Mount Google Drive & Configure Paths

Mount your Google Drive and set up the configuration for training.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ==================== CONFIGURATION ====================
# 🔧 FULL CONFIG FOR GITHUB CODESPACES (More Resources)

CONFIG = {
    # === PATHS ===
    "dataset_path": "/content/drive/MyDrive/SmartCarePlus/datasets/guardian",
    "model_save_path": "/content/drive/MyDrive/SmartCarePlus/models/guardian_movinet",
    "checkpoint_path": "/content/drive/MyDrive/SmartCarePlus/checkpoints",
    
    # === VIDEO PROCESSING (FULL RESOLUTION) ===
    "num_frames": 32,          # Full 32 frames for better temporal coverage
    "frame_height": 224,       # Full resolution
    "frame_width": 224,        # Full resolution
    "channels": 3,             # RGB channels
    
    # === TRAINING (MORE EPOCHS FOR BETTER CONVERGENCE) ===
    "batch_size": 4,           # Larger batch for Codespaces
    "epochs_phase1": 15,       # ⬆️ Increased from 5 to 15
    "epochs_phase2": 30,       # ⬆️ Increased from 10 to 30
    "learning_rate_phase1": 1e-3,
    "learning_rate_phase2": 1e-5,
    "weight_decay": 1e-5,
    
    # === DATA SPLIT ===
    "test_split": 0.15,        # 15% for testing
    "val_split": 0.15,         # 15% for validation
    "random_seed": 42,
}

# Create directories
os.makedirs(CONFIG["model_save_path"], exist_ok=True)
os.makedirs(CONFIG["checkpoint_path"], exist_ok=True)

print("✅ Full configuration loaded for GitHub Codespaces!")
print(f"📁 Dataset path: {CONFIG['dataset_path']}")
print(f"💾 Model save path: {CONFIG['model_save_path']}")
print(f"\n🚀 Training settings:")
print(f"   • Frames: {CONFIG['num_frames']}")
print(f"   • Resolution: {CONFIG['frame_height']}x{CONFIG['frame_width']}")
print(f"   • Batch size: {CONFIG['batch_size']}")
print(f"   • Phase 1 epochs: {CONFIG['epochs_phase1']}")
print(f"   • Phase 2 epochs: {CONFIG['epochs_phase2']}")

## Section 3: Load and Inspect Dataset Structure

Discover classes from folder structure and compute class weights for handling imbalance.

In [ ]:
def discover_dataset(dataset_path: str) -> Tuple[List[str], List[int], Dict[str, int], Dict[int, str]]:
    """
    Scan the dataset directory and create file lists with labels.
    
    Returns:
        video_paths: List of video file paths
        labels: List of integer labels
        class_to_idx: Dictionary mapping class names to indices
        idx_to_class: Dictionary mapping indices to class names
    """
    dataset_path = Path(dataset_path)
    
    if not dataset_path.exists():
        raise ValueError(f"Dataset path does not exist: {dataset_path}")
    
    # Get all class folders (sorted for consistency)
    class_folders = sorted([d for d in dataset_path.iterdir() if d.is_dir()])
    
    if len(class_folders) == 0:
        raise ValueError(f"No class folders found in: {dataset_path}")
    
    # Create class mappings
    class_to_idx = {folder.name: idx for idx, folder in enumerate(class_folders)}
    idx_to_class = {idx: name for name, idx in class_to_idx.items()}
    
    video_paths = []
    labels = []
    
    # Valid video extensions
    video_extensions = {'.mp4', '.avi', '.mov', '.mkv', '.MP4', '.AVI', '.MOV', '.MKV'}
    
    for class_folder in class_folders:
        class_name = class_folder.name
        class_idx = class_to_idx[class_name]
        
        # Find all video files in this class folder
        for video_file in class_folder.iterdir():
            if video_file.suffix in video_extensions:
                video_paths.append(str(video_file))
                labels.append(class_idx)
    
    return video_paths, labels, class_to_idx, idx_to_class


def compute_class_weights_dict(labels: List[int], class_to_idx: Dict[str, int]) -> Dict[int, float]:
    """Compute class weights to handle imbalanced dataset."""
    unique_classes = np.unique(labels)
    weights = compute_class_weight('balanced', classes=unique_classes, y=labels)
    return dict(zip(unique_classes, weights))


# Discover dataset
print("🔍 Scanning dataset directory...")
video_paths, labels, class_to_idx, idx_to_class = discover_dataset(CONFIG["dataset_path"])
NUM_CLASSES = len(class_to_idx)

# Compute class weights
class_weights = compute_class_weights_dict(labels, class_to_idx)

# Display statistics
print(f"\n{'='*60}")
print(f"📊 DATASET STATISTICS")
print(f"{'='*60}")
print(f"Total videos: {len(video_paths)}")
print(f"Number of classes: {NUM_CLASSES}")
print(f"\n🏷️  Class Distribution:")
for class_name, idx in class_to_idx.items():
    count = labels.count(idx)
    weight = class_weights[idx]
    print(f"   {class_name:20s}: {count:4d} videos ({100*count/len(labels):5.1f}%) | weight: {weight:.3f}")

print(f"\n✅ Found {NUM_CLASSES} classes")

## Section 4: Create Video Data Loading Pipeline

Implement efficient video loading using OpenCV with uniform frame sampling.

In [ ]:
def load_video_frames(video_path: str, 
                      num_frames: int = 32, 
                      height: int = 224,
                      width: int = 224) -> np.ndarray:
    """
    Load and uniformly sample frames from a video file.
    
    Args:
        video_path: Path to video file
        num_frames: Number of frames to sample
        height: Target frame height
        width: Target frame width
    
    Returns:
        numpy array of shape (num_frames, height, width, 3)
    """
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"⚠️ Cannot open video: {video_path}")
        return np.zeros((num_frames, height, width, 3), dtype=np.float32)
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Calculate frame indices to sample (uniform sampling)
    if total_frames >= num_frames:
        indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    else:
        # If video has fewer frames, repeat last frame
        indices = list(range(total_frames))
        if total_frames > 0:
            indices.extend([total_frames - 1] * (num_frames - total_frames))
        else:
            indices = [0] * num_frames
        indices = np.array(indices[:num_frames])
    
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        
        if ret:
            # Convert BGR to RGB
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            # Resize
            frame = cv2.resize(frame, (width, height))
            # Normalize to [0, 1]
            frame = frame.astype(np.float32) / 255.0
            frames.append(frame)
        else:
            # If frame read fails, use last valid frame or zeros
            if frames:
                frames.append(frames[-1].copy())
            else:
                frames.append(np.zeros((height, width, 3), dtype=np.float32))
    
    cap.release()
    
    # Ensure we have exactly num_frames
    while len(frames) < num_frames:
        frames.append(frames[-1].copy() if frames else np.zeros((height, width, 3), dtype=np.float32))
    
    return np.array(frames[:num_frames], dtype=np.float32)


# Test video loading
print("🎬 Testing video loading...")
test_video_path = video_paths[0]
test_frames = load_video_frames(
    test_video_path, 
    CONFIG["num_frames"], 
    CONFIG["frame_height"], 
    CONFIG["frame_width"]
)
print(f"   Video: {os.path.basename(test_video_path)}")
print(f"   Frames shape: {test_frames.shape}")
print(f"   Value range: [{test_frames.min():.3f}, {test_frames.max():.3f}]")
print("✅ Video loading works!")

## Section 5: Data Augmentation

Apply spatial and temporal augmentations to improve model robustness.

In [ ]:
def apply_augmentation(frames: np.ndarray, training: bool = True) -> np.ndarray:
    """
    Apply data augmentation to video frames.
    
    Augmentations applied:
    - Random horizontal flip (50% chance)
    - Random brightness adjustment
    - Random contrast adjustment
    - Random temporal jittering
    
    Args:
        frames: numpy array of shape (num_frames, H, W, C)
        training: Whether in training mode
    
    Returns:
        Augmented frames
    """
    if not training:
        return frames
    
    frames = frames.copy()
    
    # Random horizontal flip (50% chance)
    if random.random() > 0.5:
        frames = frames[:, :, ::-1, :]
    
    # Random brightness adjustment (±20%)
    brightness_factor = random.uniform(0.8, 1.2)
    frames = np.clip(frames * brightness_factor, 0, 1)
    
    # Random contrast adjustment (±20%)
    contrast_factor = random.uniform(0.8, 1.2)
    mean = np.mean(frames, axis=(1, 2, 3), keepdims=True)
    frames = np.clip((frames - mean) * contrast_factor + mean, 0, 1)
    
    # Random temporal jittering (shift frames by ±2)
    if random.random() > 0.5:
        shift = random.randint(-2, 2)
        if shift != 0:
            frames = np.roll(frames, shift, axis=0)
    
    return frames.astype(np.float32)


# Visualize augmentation
print("🎨 Visualizing augmentation...")
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Calculate frame step based on actual number of frames
num_frames = CONFIG["num_frames"]
frame_step = max(1, num_frames // 4)  # Show 4 evenly spaced frames

# Original frames
for i in range(4):
    frame_idx = min(i * frame_step, num_frames - 1)  # Prevent out of bounds
    axes[0, i].imshow(test_frames[frame_idx])
    axes[0, i].set_title(f"Original Frame {frame_idx}")
    axes[0, i].axis('off')

# Augmented frames
aug_frames = apply_augmentation(test_frames, training=True)
for i in range(4):
    frame_idx = min(i * frame_step, num_frames - 1)
    axes[1, i].imshow(aug_frames[frame_idx])
    axes[1, i].set_title(f"Augmented Frame {frame_idx}")
    axes[1, i].axis('off')

plt.suptitle(f"Data Augmentation Example ({num_frames} frames)", fontsize=14)
plt.tight_layout()
plt.show()
print("✅ Augmentation configured!")

## Section 6: Create tf.data Dataset Pipeline

Build efficient data pipeline with tf.data for training.

In [ ]:
def create_dataset(video_paths: List[str], 
                   labels: List[int], 
                   training: bool = True,
                   batch_size: int = 4) -> tf.data.Dataset:
    """
    Create a tf.data.Dataset for efficient data loading.
    
    Args:
        video_paths: List of video file paths
        labels: List of integer labels
        training: Whether this is training data
        batch_size: Batch size
    
    Returns:
        tf.data.Dataset yielding (frames, labels)
    """
    
    def load_video_tf(video_path, label):
        """TensorFlow wrapper for video loading."""
        def _load(path, lbl):
            path_str = path.numpy().decode('utf-8')
            frames = load_video_frames(
                path_str,
                CONFIG["num_frames"],
                CONFIG["frame_height"],
                CONFIG["frame_width"]
            )
            
            if training:
                frames = apply_augmentation(frames, training=True)
            
            return frames.astype(np.float32), lbl.numpy()
        
        frames, lbl = tf.py_function(
            _load, 
            [video_path, label], 
            [tf.float32, tf.int32]
        )
        
        # Set shapes explicitly (required for model)
        frames.set_shape([CONFIG["num_frames"], 
                          CONFIG["frame_height"], 
                          CONFIG["frame_width"], 
                          CONFIG["channels"]])
        lbl.set_shape([])
        
        return frames, lbl
    
    # Create dataset from tensor slices
    dataset = tf.data.Dataset.from_tensor_slices((video_paths, labels))
    
    if training:
        dataset = dataset.shuffle(buffer_size=len(video_paths), seed=CONFIG["random_seed"])
    
    # Map the loading function
    dataset = dataset.map(
        load_video_tf,
        num_parallel_calls=tf.data.AUTOTUNE
    )
    
    # Batch and prefetch
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset


# Split dataset into train/val/test
print("📊 Splitting dataset...")

# First split: train+val vs test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    video_paths, labels,
    test_size=CONFIG["test_split"],
    stratify=labels,
    random_state=CONFIG["random_seed"]
)

# Second split: train vs val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=CONFIG["val_split"] / (1 - CONFIG["test_split"]),
    stratify=y_train_val,
    random_state=CONFIG["random_seed"]
)

print(f"\n📊 Data Split:")
print(f"   Training:   {len(X_train):4d} videos ({100*len(X_train)/len(video_paths):.1f}%)")
print(f"   Validation: {len(X_val):4d} videos ({100*len(X_val)/len(video_paths):.1f}%)")
print(f"   Test:       {len(X_test):4d} videos ({100*len(X_test)/len(video_paths):.1f}%)")

# Create datasets
train_dataset = create_dataset(X_train, y_train, training=True, batch_size=CONFIG["batch_size"])
val_dataset = create_dataset(X_val, y_val, training=False, batch_size=CONFIG["batch_size"])
test_dataset = create_dataset(X_test, y_test, training=False, batch_size=CONFIG["batch_size"])

print("\n✅ Datasets created!")

## Section 7: Build Video Classification Backbone

We use **EfficientNetV2-S + LSTM** architecture for video classification:

- **EfficientNetV2-S**: State-of-the-art image backbone pretrained on ImageNet
- **TimeDistributed**: Apply EfficientNet to each frame independently
- **Bidirectional LSTM**: Capture temporal patterns in video sequences

This approach is more stable than MoViNet with TensorFlow 2.15+ and provides excellent performance for:
- Fall detection (captures motion patterns across frames)
- Gait analysis (temporal gait features)
- Activity recognition (ADL classification)

**Note**: Weights are automatically downloaded from Keras Applications (ImageNet pretrained).

In [ ]:
# Ensure CONFIG is defined (in case cells were run out of order)
if 'CONFIG' not in globals():
    CONFIG = {
        "num_frames": 32,
        "frame_height": 224,
        "frame_width": 224,
        "channels": 3,
        "batch_size": 4,
        "dataset_path": "/content/drive/MyDrive/SmartCarePlus/datasets/guardian",
        "model_save_path": "/content/drive/MyDrive/SmartCarePlus/models/guardian_movinet",
        "checkpoint_path": "/content/drive/MyDrive/SmartCarePlus/checkpoints",
        "epochs_phase1": 15,
        "epochs_phase2": 30,
        "learning_rate_phase1": 1e-3,
        "learning_rate_phase2": 1e-5,
        "weight_decay": 1e-5,
        "test_split": 0.15,
        "val_split": 0.15,
        "random_seed": 42,
    }
    print("⚠️ CONFIG was not defined - using default values")

# Ensure idx_to_class and NUM_CLASSES are defined
if 'idx_to_class' not in globals():
    print("⚠️ Run Section 3 first to discover dataset!")


def build_video_backbone():
    """
    Build a powerful video classification backbone using EfficientNetV2-S.
    
    Architecture:
    1. TimeDistributed EfficientNetV2-S (per-frame features)
    2. Global Average Pooling over spatial dimensions
    3. Bidirectional LSTM for temporal modeling
    4. Dense feature vector output
    
    Returns:
        Keras Model that outputs feature vectors
    """
    from tensorflow.keras import layers, models
    from tensorflow.keras.applications import EfficientNetV2S
    
    print("📦 Building Video Classification Backbone...")
    print("   Using EfficientNetV2-S + Bidirectional LSTM")
    
    # Input: (batch, frames, height, width, channels)
    input_shape = (
        CONFIG["num_frames"],
        CONFIG["frame_height"],
        CONFIG["frame_width"],
        CONFIG["channels"]
    )
    
    # Load EfficientNetV2-S pretrained on ImageNet
    print("   Loading EfficientNetV2-S (ImageNet pretrained)...")
    efficientnet = EfficientNetV2S(
        include_top=False,
        weights='imagenet',
        input_shape=(CONFIG["frame_height"], CONFIG["frame_width"], CONFIG["channels"]),
        pooling='avg'  # Global average pooling
    )
    
    # Freeze most layers initially (will unfreeze in Phase 2)
    # Keep last 30 layers trainable for better feature adaptation
    for layer in efficientnet.layers[:-30]:
        layer.trainable = False
    
    print(f"   EfficientNetV2 output shape: {efficientnet.output_shape}")
    
    # Build video model
    video_input = layers.Input(shape=input_shape, name="video_input")
    
    # Apply EfficientNet to each frame using TimeDistributed
    # Output: (batch, frames, features)
    frame_features = layers.TimeDistributed(
        efficientnet, 
        name="frame_encoder"
    )(video_input)
    
    print(f"   Frame features shape: (batch, {CONFIG['num_frames']}, {efficientnet.output_shape[-1]})")
    
    # Temporal modeling with Bidirectional LSTM
    x = layers.Bidirectional(
        layers.LSTM(256, return_sequences=True, dropout=0.3, recurrent_dropout=0.2),
        name="temporal_lstm_1"
    )(frame_features)
    
    x = layers.Bidirectional(
        layers.LSTM(128, return_sequences=False, dropout=0.3, recurrent_dropout=0.2),
        name="temporal_lstm_2"
    )(x)
    
    # Feature vector output with BatchNorm for stability
    features = layers.Dense(512, activation='relu', name="feature_dense")(x)
    features = layers.BatchNormalization(name="feature_bn")(features)
    features = layers.Dropout(0.4, name="feature_dropout")(features)
    
    # Build model
    backbone = models.Model(
        inputs=video_input, 
        outputs=features, 
        name="video_backbone"
    )
    
    print(f"✅ Video backbone built!")
    print(f"   Input shape: {input_shape}")
    print(f"   Output shape: (batch, 512)")
    print(f"   Total params: {backbone.count_params():,}")
    print(f"   Trainable params: {sum([tf.keras.backend.count_params(w) for w in backbone.trainable_weights]):,}")
    
    return backbone


# Clear any previous models from memory
import gc
tf.keras.backend.clear_session()
gc.collect()

# Build the backbone
base_model = build_video_backbone()
base_model.summary()

In [ ]:
# Verify backbone model is ready
# EfficientNetV2-S weights are automatically downloaded from Keras Applications
# No manual weight download needed!

print("✅ Backbone model ready!")
print(f"\n📊 Model Architecture Summary:")
print(f"   • Base: EfficientNetV2-S (ImageNet pretrained)")
print(f"   • Temporal: Bidirectional LSTM (2 layers)")
print(f"   • Output: 512-dimensional feature vector")
print(f"\n🎯 This architecture is excellent for:")
print(f"   • Fall detection (captures motion patterns)")
print(f"   • Gait analysis (temporal gait features)")
print(f"   • Activity recognition (ADL classification)")
print(f"   • TUG test assessment (timing + movement)")

# Quick test: verify model accepts correct input shape
print(f"\n🧪 Testing model with random input...")
test_input = tf.random.normal([1, CONFIG["num_frames"], CONFIG["frame_height"], CONFIG["frame_width"], CONFIG["channels"]])
test_output = base_model(test_input, training=False)
print(f"   Input shape:  {test_input.shape}")
print(f"   Output shape: {test_output.shape}")
print(f"✅ Model test passed!")

## Section 8: Build Classification Model

Add a custom classification head for our Guardian activity classes. Uses dropout for regularization and a dense layer for class predictions.

In [ ]:
# Import Keras layers and models if not already imported
from tensorflow.keras import layers, models

def build_guardian_model(backbone, num_classes: int, dropout_rate: float = 0.3):
    """
    Build the complete Guardian classification model.
    
    Args:
        backbone: Video backbone model (EfficientNetV2 + LSTM)
        num_classes: Number of activity classes
        dropout_rate: Dropout rate for regularization
    
    Returns:
        Complete Keras model for video classification
    """
    print(f"🏗️ Building Guardian classification model...")
    print(f"   Number of classes: {num_classes}")
    print(f"   Dropout rate: {dropout_rate}")
    
    # Get backbone output (already has input defined)
    features = backbone.output
    
    # Classification head with more capacity
    x = layers.Dropout(dropout_rate, name="classifier_dropout")(features)
    x = layers.Dense(256, activation='relu', name="classifier_dense_1")(x)
    x = layers.BatchNormalization(name="classifier_bn")(x)
    x = layers.Dropout(dropout_rate / 2, name="classifier_dropout_2")(x)
    x = layers.Dense(128, activation='relu', name="classifier_dense_2")(x)
    
    # Output layer with float32 for numerical stability with mixed precision
    outputs = layers.Dense(
        num_classes, 
        activation="softmax",
        dtype='float32',  # Ensure output is float32 for loss calculation
        kernel_regularizer=tf.keras.regularizers.l2(0.001),
        name="predictions"
    )(x)
    
    # Build model using backbone's input
    model = models.Model(
        inputs=backbone.input, 
        outputs=outputs, 
        name="guardian_video_classifier"
    )
    
    print("✅ Model built successfully!")
    
    return model


# Create class_names list from idx_to_class (from Section 3)
if 'idx_to_class' in globals():
    class_names = [idx_to_class[i] for i in range(len(idx_to_class))]
    NUM_CLASSES = len(class_names)
else:
    print("⚠️ idx_to_class not found! Run Section 3 first.")
    # Default for testing
    class_names = ['ADL', 'Fall', 'arthritis_gait', 'good_gait', 'tug']
    NUM_CLASSES = 5

# Build the model
model = build_guardian_model(base_model, NUM_CLASSES)

# Print model summary
print(f"\n📊 Model Architecture:")
print(f"   Classes: {class_names}")
print(f"   Total parameters: {model.count_params():,}")

# Estimate memory usage
param_memory_mb = model.count_params() * 4 / (1024 * 1024)  # 4 bytes per float32
print(f"   Estimated param memory: {param_memory_mb:.1f} MB")
model.summary()

## Section 9: Compute Class Weights & Compile Model

Handle class imbalance by computing class weights. Use label smoothing for better generalization.

In [ ]:
# Compute class weights to handle imbalance
print("⚖️ Computing class weights...")
class_weight_values = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)
class_weights = dict(enumerate(class_weight_values))

print("\nClass weights:")
for idx, (name, weight) in enumerate(zip(class_names, class_weight_values)):
    count = labels.count(idx)
    print(f"   {name}: {weight:.3f} (n={count})")


# Create warmup + cosine decay learning rate schedule
class WarmUpCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Learning rate schedule with warmup and cosine decay."""
    
    def __init__(self, initial_lr, warmup_steps, total_steps, min_lr=1e-7):
        super().__init__()
        self.initial_lr = initial_lr
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr
    
    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        
        # Warmup phase
        warmup_lr = self.initial_lr * (step / self.warmup_steps)
        
        # Cosine decay phase
        decay_steps = self.total_steps - self.warmup_steps
        decay_step = step - self.warmup_steps
        cosine_decay = 0.5 * (1 + tf.cos(np.pi * decay_step / decay_steps))
        cosine_lr = self.min_lr + (self.initial_lr - self.min_lr) * cosine_decay
        
        # Use warmup or cosine based on step
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)
    
    def get_config(self):
        return {
            "initial_lr": self.initial_lr,
            "warmup_steps": self.warmup_steps,
            "total_steps": self.total_steps,
            "min_lr": self.min_lr
        }


# Calculate training steps
steps_per_epoch = max(1, len(X_train) // CONFIG["batch_size"])
total_steps = steps_per_epoch * CONFIG["epochs_phase1"]  # Fixed: epochs_phase1
warmup_steps = steps_per_epoch * 2  # 2 epochs warmup

print(f"\n📈 Learning Rate Schedule:")
print(f"   Steps per epoch: {steps_per_epoch}")
print(f"   Total steps (phase 1): {total_steps}")
print(f"   Warmup steps: {warmup_steps}")

# Create learning rate schedule
lr_schedule = WarmUpCosineDecay(
    initial_lr=CONFIG["learning_rate_phase1"],  # Fixed: learning_rate_phase1
    warmup_steps=warmup_steps,
    total_steps=total_steps
)

# Compile with mixed precision
print("\n⚙️ Compiling model...")
optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

# Label smoothing loss
loss = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=False  # We use softmax in output layer
)

model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=[
        'accuracy',
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name='top2_accuracy')
    ]
)

print("✅ Model compiled successfully!")

## Section 10: Setup Training Callbacks

Configure callbacks for:
- Early stopping with patience
- Model checkpoint (save best)
- Learning rate reduction on plateau
- TensorBoard logging

In [ ]:
# Import datetime for timestamp
from datetime import datetime

# Create output directories
model_dir = Path(CONFIG["model_save_path"])
log_dir = Path("/content/drive/MyDrive/SmartCarePlus/logs")
model_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

# Timestamp for this run
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Patience for early stopping - increased for more epochs
patience = 10  # ⬆️ Increased from 5 to 10

# Define callbacks
callbacks = [
    # Early stopping - monitor val_accuracy
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=patience,
        restore_best_weights=True,
        verbose=1,
        mode='max'
    ),
    
    # Model checkpoint - save best model
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(model_dir / f"guardian_model_{timestamp}_best.keras"),
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=False,
        verbose=1,
        mode='max'
    ),
    
    # Reduce LR on plateau
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,  # ⬆️ Increased from 3 to 5
        min_lr=1e-7,
        verbose=1
    ),
    
    # TensorBoard logging
    tf.keras.callbacks.TensorBoard(
        log_dir=str(log_dir / f"guardian_{timestamp}"),
        histogram_freq=1,
        write_graph=True,
        update_freq='epoch'
    )
]

print(f"✅ Callbacks configured:")
print(f"   📁 Model checkpoint: {model_dir}")
print(f"   📊 TensorBoard logs: {log_dir}")
print(f"   ⏱️ Run timestamp: {timestamp}")
print(f"   ⏳ Early stopping patience: {patience} epochs")
print(f"   🎯 Total epochs: Phase1={CONFIG['epochs_phase1']} + Phase2={CONFIG['epochs_phase2']}")

## Section 11: Two-Phase Training

**Phase 1:** Train only the classification head (base model frozen)  
**Phase 2:** Fine-tune the entire model with a lower learning rate

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 1: Train Classification Head Only (Base Frozen)
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("🎯 PHASE 1: Training Classification Head (EfficientNet Frozen)")
print("=" * 70)

# Safety check: ensure required variables exist
if 'base_model' not in globals():
    raise NameError("❌ 'base_model' not defined! Run Section 7 first to build the backbone.")
if 'model' not in globals():
    raise NameError("❌ 'model' not defined! Run Section 8 first to build the classifier.")
if 'train_dataset' not in globals():
    raise NameError("❌ 'train_dataset' not defined! Run Section 6 first to create datasets.")
if 'class_weights' not in globals():
    raise NameError("❌ 'class_weights' not defined! Run Section 3 first to discover dataset.")

# Freeze the EfficientNet backbone layers in base_model
# The TimeDistributed layer wraps EfficientNet
for layer in base_model.layers:
    if 'frame_encoder' in layer.name:
        layer.trainable = False
    # Keep LSTM and Dense layers trainable
    
print(f"\n📊 Trainable layers:")
trainable_count = 0
for layer in model.layers:
    if layer.trainable:
        trainable_count += 1
        if hasattr(layer, 'trainable_weights') and layer.trainable_weights:
            print(f"   • {layer.name}")
            
# Recompile with frozen base
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG["learning_rate_phase1"]),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name='top2_accuracy')]
)

print(f"\n   Training for {CONFIG['epochs_phase1']} epochs...")
print()

# Train Phase 1
history_phase1 = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=CONFIG["epochs_phase1"],
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)

print("\n✅ Phase 1 complete!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 2: Fine-tune Entire Model
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("🔧 PHASE 2: Fine-tuning Entire Model (EfficientNet Unfrozen)")
print("=" * 70)

# Unfreeze the EfficientNet backbone
for layer in base_model.layers:
    layer.trainable = True

# Use much lower learning rate for fine-tuning
fine_tune_lr = CONFIG["learning_rate_phase2"]

# Recompile with lower LR
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=fine_tune_lr),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name='top2_accuracy')]
)

# Count trainable params
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
print(f"\n📊 Trainable parameters: {trainable_params:,}")
print(f"   Fine-tuning learning rate: {fine_tune_lr}")
print(f"   Training for {CONFIG['epochs_phase2']} epochs...")
print()

# Update callbacks with new checkpoint name
callbacks_phase2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=patience,  # Use patience from Section 10
        restore_best_weights=True,
        verbose=1,
        mode='max'
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(model_dir / f"guardian_model_{timestamp}_finetune.keras"),
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=False,
        verbose=1,
        mode='max'
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-8,
        verbose=1
    ),
    tf.keras.callbacks.TensorBoard(
        log_dir=str(log_dir / f"guardian_{timestamp}_finetune"),
        histogram_freq=1,
        update_freq='epoch'
    )
]

# Continue training from where Phase 1 ended
initial_epoch = len(history_phase1.history['loss'])

history_phase2 = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=initial_epoch + CONFIG["epochs_phase2"],
    initial_epoch=initial_epoch,
    callbacks=callbacks_phase2,
    class_weight=class_weights,
    verbose=1
)

print("\n✅ Phase 2 complete!")

## Section 12: Plot Training History

Visualize training and validation metrics across both phases.

In [ ]:
def plot_training_history(history1, history2, save_path=None):
    """
    Plot training history for both phases.
    
    Args:
        history1: History from Phase 1
        history2: History from Phase 2
        save_path: Optional path to save the figure
    """
    # Combine histories
    acc = history1.history['accuracy'] + history2.history['accuracy']
    val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
    loss = history1.history['loss'] + history2.history['loss']
    val_loss = history1.history['val_loss'] + history2.history['val_loss']
    
    epochs = range(1, len(acc) + 1)
    phase1_end = len(history1.history['accuracy'])
    
    # Create figure
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy plot
    ax1 = axes[0]
    ax1.plot(epochs, acc, 'b-', label='Training Accuracy', linewidth=2)
    ax1.plot(epochs, val_acc, 'r-', label='Validation Accuracy', linewidth=2)
    ax1.axvline(x=phase1_end, color='gray', linestyle='--', label='Phase 2 Start')
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Accuracy', fontsize=12)
    ax1.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax1.legend(loc='lower right')
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim([0, 1])
    
    # Loss plot
    ax2 = axes[1]
    ax2.plot(epochs, loss, 'b-', label='Training Loss', linewidth=2)
    ax2.plot(epochs, val_loss, 'r-', label='Validation Loss', linewidth=2)
    ax2.axvline(x=phase1_end, color='gray', linestyle='--', label='Phase 2 Start')
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Loss', fontsize=12)
    ax2.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax2.legend(loc='upper right')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"📊 Figure saved to: {save_path}")
    
    plt.show()
    
    # Print best metrics
    best_val_acc = max(val_acc)
    best_epoch = val_acc.index(best_val_acc) + 1
    print(f"\n📈 Best Validation Accuracy: {best_val_acc:.4f} (Epoch {best_epoch})")


# Plot the training history
plot_training_history(
    history_phase1, 
    history_phase2,
    save_path=str(model_dir / f"training_history_{timestamp}.png")
)

## Section 13: Model Evaluation & Confusion Matrix

Evaluate on test set and generate detailed metrics including confusion matrix and classification report.

In [ ]:
# Evaluate on test set
print("🧪 Evaluating model on test set...")
test_loss, test_accuracy, test_top2_acc = model.evaluate(test_dataset, verbose=1)

print(f"\n📊 Test Results:")
print(f"   Loss:         {test_loss:.4f}")
print(f"   Accuracy:     {test_accuracy:.4f} ({test_accuracy*100:.1f}%)")
print(f"   Top-2 Acc:    {test_top2_acc:.4f} ({test_top2_acc*100:.1f}%)")


# Get predictions for confusion matrix
print("\n🔮 Generating predictions...")
y_true = []
y_pred = []

for videos, labels_batch in test_dataset:
    preds = model.predict(videos, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1).tolist())
    y_true.extend(labels_batch.numpy().tolist())

y_true = np.array(y_true)
y_pred = np.array(y_pred)


# Classification Report
print("\n📋 Classification Report:")
print("=" * 60)
report = classification_report(y_true, y_pred, target_names=class_names, digits=3)
print(report)


# Confusion Matrix
print("\n📊 Generating confusion matrix...")
cm = confusion_matrix(y_true, y_pred)

# Normalize confusion matrix
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('Actual', fontsize=12)
axes[0].set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_xlabel('Predicted', fontsize=12)
axes[1].set_ylabel('Actual', fontsize=12)
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(str(model_dir / f"confusion_matrix_{timestamp}.png"), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Confusion matrix saved!")

## Section 14: Save Final Model & Export

Save the model in multiple formats:
- `.keras` format (recommended for TensorFlow)
- SavedModel format (for TensorFlow Serving)
- TFLite format (for mobile deployment)

In [ ]:
# Save class names mapping
class_mapping = {i: name for i, name in enumerate(class_names)}
class_mapping_path = model_dir / f"class_mapping_{timestamp}.json"

with open(class_mapping_path, 'w') as f:
    json.dump(class_mapping, f, indent=2)
print(f"📁 Class mapping saved: {class_mapping_path}")


# Save in Keras format
keras_path = model_dir / f"guardian_movinet_final_{timestamp}.keras"
model.save(keras_path)
print(f"📁 Keras model saved: {keras_path}")


# Save as SavedModel (for TensorFlow Serving)
savedmodel_path = model_dir / f"guardian_movinet_savedmodel_{timestamp}"
model.export(savedmodel_path)
print(f"📁 SavedModel saved: {savedmodel_path}")


# Convert to TFLite (for mobile deployment)
print("\n📱 Converting to TFLite...")
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    
    tflite_model = converter.convert()
    
    tflite_path = model_dir / f"guardian_movinet_{timestamp}.tflite"
    with open(tflite_path, 'wb') as f:
        f.write(tflite_model)
    
    print(f"📁 TFLite model saved: {tflite_path}")
    print(f"   Size: {len(tflite_model) / (1024*1024):.1f} MB")
except Exception as e:
    print(f"⚠️ TFLite conversion failed: {e}")
    print("   (This is expected for some complex models)")


# Save training summary
summary = {
    "timestamp": timestamp,
    "model_type": "MoViNet-A2",
    "num_classes": len(class_names),
    "class_names": class_names,
    "config": CONFIG,
    "training": {
        "phase1_epochs": len(history_phase1.history['loss']),
        "phase2_epochs": len(history_phase2.history['loss']),
        "best_val_accuracy": float(max(history_phase1.history['val_accuracy'] + 
                                       history_phase2.history['val_accuracy']))
    },
    "test_results": {
        "loss": float(test_loss),
        "accuracy": float(test_accuracy),
        "top2_accuracy": float(test_top2_acc)
    },
    "data_split": {
        "train": len(X_train),
        "val": len(X_val),
        "test": len(X_test)
    }
}

summary_path = model_dir / f"training_summary_{timestamp}.json"
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n📋 Training summary saved: {summary_path}")


# Print final summary
print("\n" + "=" * 70)
print("🎉 TRAINING COMPLETE!")
print("=" * 70)
print(f"\n📊 Final Results:")
print(f"   Test Accuracy: {test_accuracy*100:.1f}%")
print(f"   Test Top-2:    {test_top2_acc*100:.1f}%")
print(f"\n📁 Saved Files:")
print(f"   • {keras_path}")
print(f"   • {savedmodel_path}")
print(f"   • {class_mapping_path}")
print(f"   • {summary_path}")
print(f"\n💡 Next Steps:")
print(f"   1. Download the model from Google Drive")
print(f"   2. Place in backend/ml_models/guardian/")
print(f"   3. Update model loading in guardian_service")